# Modelización Matemática - Parcial 1
## Diego Ángel Gallardo Costilla

In [43]:
import pandas as pd

df = pd.read_csv("TablaParcial1.csv", sep=";", index_col=0)
df

,Acción,Aventura,Ci-Fi,Superh,Anim,Rom,Fant,Drama
Avatar,1,1,1,0,0,1,0,1
Avengers,1,0,1,1,0,0,0,1
Ne Zha,0,1,0,0,1,1,1,0
Star Wars VII,1,1,1,0,0,0,0,1
Spiderman,1,0,1,1,0,0,0,1
LOTR,1,1,0,0,0,0,1,1


In [44]:
def extension(df, atributos):
    """Dado un DataFrame y una lista de atributos, 
    devuelve solo los objetos que poseen 
    todos los atributos especificados."""
    if not atributos:
        return df.index.tolist()
    for atributo in atributos:
        if atributo in df.columns:
            df = df[df[atributo] == 1]
        else:
            print(f"El atributo '{atributo}' no se encuentra en el DataFrame.")
    return df.index.tolist()

In [45]:
def intension(df, objetos):
    """Dado un DataFrame y una lista de objetos, 
    devuelve solo los atributos que todos los objetos 
    especificados poseen."""
    filasobjetos = []
    for objeto in objetos:
        if objeto in df.index:
            # Extrae la fila completa del objeto y la añade a la lista de filasobjetos
            filasobjetos.append(df.loc[df.index == objeto])
        else:
            print(f"El objeto '{objeto}' no se encuentra en el DataFrame.")
    if not filasobjetos:
        return df.columns.tolist()  # Si no se encontraron objetos, devolvemos todos los atributos
    # Concatenamos todas las filas de filasobjetos en un solo DataFrame
    dfobj = pd.concat(filasobjetos)
    # Seleccionamos solo las columnas donde todos los valores son 1, es decir, los atributos comunes a todos los objetos
    atributos_comunes = dfobj.columns[(dfobj == 1).all()]
    return atributos_comunes.tolist()

In [46]:
def intext(df, objetos):
    return extension(df, intension(df, objetos))
def extint(df, atributos):
    return intension(df, extension(df, atributos))

In [47]:
def algoritmo(df):
    '''Dado un contexto, devuelve todos sus conceptos de forma "óptima" (o al menos como yo lo he intentado).'''
    conceptos = []
    objetos = df.index.tolist()
    atributos_restantes = [col for col in df.columns]
    
    # Empezamos definiendo el primer concepto, el formado por todos los objetos y los atributos comunes a todos ellos
    # A partir de ahí, se van generando nuevos conceptos a partir de los atributos con mayor extensión, y así sucesivamente
    atributos_primer_concepto = []
    # Para el primer concepto, solo se cogen aquellos atributos cuya extensión son todos los objetos
    for atributo in list(atributos_restantes):
        if len(extension(df, [atributo])) == len(objetos):
            atributos_primer_concepto.append(atributo)
            atributos_restantes.remove(atributo)
            
    conceptos.append([objetos, atributos_primer_concepto])
    
    # BUCLE PRINCIPAL
    while atributos_restantes:
        # Cogemos el atributo de mayor extensión de los que quedan
        atributo_max = max(atributos_restantes, key=lambda x: len(extension(df, [x])))
        objetos_nuevo = extension(df, [atributo_max])
        
        # Lo quitamos de la lista para que el while avance
        atributos_restantes.remove(atributo_max)
        
        # Empezamos la lista de nuevos conceptos que se van a generar en esta iteración, para luego añadirlos todos juntos al final de la iteración
        nuevos_conceptos_iteracion = []
        
        # Cruzamos la extensión de este nuevo atributo con TODOS los conceptos que ya teníamos
        for concepto in conceptos:
            # Intersecamos los objetos
            interseccion_obj = list(set(objetos_nuevo) & set(concepto[0]))
            
            # Para poder comparar listas de objetos fácilmente, es bueno ordenarlas
            interseccion_obj.sort()
            
            # Verificamos si esta intersección ha generado un grupo de objetos que NO teníamos antes
            conceptos_existentes = [sorted(c[0]) for c in conceptos]
            conceptos_nuevos = [sorted(c[0]) for c in nuevos_conceptos_iteracion]
            
            if (interseccion_obj not in conceptos_existentes) and (interseccion_obj not in conceptos_nuevos):
                # Como es un concepto nuevo, calculamos TODOS sus atributos (su intensión)
                atributos_interseccion = intension(df, interseccion_obj)
                nuevos_conceptos_iteracion.append([interseccion_obj, atributos_interseccion])
        
        # Añadimos los conceptos descubiertos en esta iteración a la lista global
        conceptos.extend(nuevos_conceptos_iteracion)
        
    return conceptos

In [48]:
def infimo_irreducibles(df):
    '''Devuelve los conceptos que se consideran ínfimo irreducibles.'''
    conceptos = algoritmo(df)
    
    # Un concepto es ínfimo irreducible si, para empezar, su extensión no es el total:
    infimoirr = [c for c in conceptos if len(c[0]) < len(df.index)]
    
    # Creamos una lista temporal para los que vamos a eliminar
    a_eliminar = []
    
    for concepto in infimoirr:
        # Hallamos los conceptos que lo contienen (su extensión está contenida en la extensión de esos conceptos)
        conceptos_que_lo_contienen = [c for c in conceptos if set(concepto[0]).issubset(set(c[0])) and c != concepto]
        
        if not conceptos_que_lo_contienen:  # Si no hay conceptos que lo contengan, no puede ser generado
            continue
            
        # Iniciamos la intersección con el primer concepto que lo contiene
        int_conceptos_que_lo_contienen = set(conceptos_que_lo_contienen[0][0])
        
        # Intersecamos con los demás conceptos que lo contienen
        for c in conceptos_que_lo_contienen[1:]:
            int_conceptos_que_lo_contienen = int_conceptos_que_lo_contienen & set(c[0])
        
        # Si la intersección es igual a la extensión del concepto, entonces se puede generar
        if set(concepto[0]) == int_conceptos_que_lo_contienen:
            a_eliminar.append(concepto)
    
    # Eliminamos los que no son irreducibles
    for concepto in a_eliminar:
        infimoirr.remove(concepto)
    
    return infimoirr

In [49]:
def clasificar(data):
    mf = [set(exten[0]) for exten in infimo_irreducibles(data)]
    
    atr_nec = []
    atr_rel = []
    atr_in = []

    for atr in data.columns:
        ext = set(extension(data, [atr]))
        gen_atrib = [atrib for atrib in data.columns if atrib != atr]
        gen_exten = [set(extension(data, [atributo])) for atributo in gen_atrib]
        if ext not in mf:
            atr_in.append(atr)
        elif ext in mf and ext in gen_exten:    
            atr_rel.append(atr)
        elif ext in mf and ext not in gen_exten:
            atr_nec.append(atr)
    return print('\n', 'Atributos absolutamente necesarios: ', atr_nec, '\n', 'Atributos relativamente necesarios: ', atr_rel, '\n', 'Atributos absolutamente innecesarios: ', atr_in)

In [50]:
clasificar(df)


 Atributos absolutamente necesarios:  ['Aventura', 'Ci-Fi', 'Superh', 'Rom', 'Fant'] 
 Atributos relativamente necesarios:  ['Acción', 'Drama'] 
 Atributos absolutamente innecesarios:  ['Anim']


In [51]:
def es_valida(df, atributo1, atributo2):
    '''Dado un contexto y dos grupos de atributos, devuelve si el primero implica al segundo.'''
    atr1 = set(extint(df, atributo1))
    atr2 = set(atributo2)
    return atr2.issubset(atr1)

In [54]:
es_valida(df,["Ci-Fi"],["Acción"])

True

In [52]:
es_valida(df,["Ci-Fi","Superh"],["Acción"])

True

In [53]:
df2 = pd.read_csv("TablaParcial1 copy.csv", sep=";", index_col=0)
df2

,Acción,Aventura,Ci-Fi,Superh,Anim,Rom,Fant,Drama
Avatar,1,1,1,0,0,1,0,1
Avengers,1,0,1,1,0,0,0,1
Ne Zha,0,1,0,0,1,1,1,0
Star Wars VII,1,1,1,0,0,0,0,1
Spiderman,1,0,1,1,0,0,0,1
LOTR,1,1,0,0,0,0,1,1
Titanic,0,0,0,0,0,1,0,1
Zootopia 2,0,1,0,0,1,0,1,0
Jurassic World,1,1,1,0,0,0,0,0
Inside Out 2,0,0,0,0,1,0,1,1


In [56]:
clasificar(df2)


 Atributos absolutamente necesarios:  ['Acción', 'Aventura', 'Ci-Fi', 'Superh', 'Anim', 'Rom', 'Fant', 'Drama'] 
 Atributos relativamente necesarios:  [] 
 Atributos absolutamente innecesarios:  []


In [57]:
def supremo_irreducibles(df):
    '''Devuelve los conceptos que se consideran supremo irreducibles.'''
    conceptos = algoritmo(df)
    
    # Un concepto es supremo irreducible si, para empezar, su extensión no es el total:
    supremoirr = [c for c in conceptos if len(c[0]) < len(df.index)]
    
    # Creamos una lista temporal para los que vamos a eliminar
    a_eliminar = []
    
    for concepto in supremoirr:
        # Hallamos los conceptos que lo contienen (su extensión está contenida en la extensión de esos conceptos)
        conceptos_que_lo_contienen = [c for c in conceptos if set(concepto[0]).issubset(set(c[0])) and c != concepto]
        
        if not conceptos_que_lo_contienen:  # Si no hay conceptos que lo contengan, no puede ser generado
            continue
            
        # Iniciamos la intersección con el primer concepto que lo contiene
        int_conceptos_que_lo_contienen = set(conceptos_que_lo_contienen[0][0])
        
        # Intersecamos con los demás conceptos que lo contienen
        for c in conceptos_que_lo_contienen[1:]:
            int_conceptos_que_lo_contienen = int_conceptos_que_lo_contienen & set(c[0])
        
        # Si la intersección es igual a la extensión del concepto, entonces se puede generar
        if set(concepto[0]) == int_conceptos_que_lo_contienen:
            a_eliminar.append(concepto)
    
    # Eliminamos los que no son irreducibles
    for concepto in a_eliminar:
        supremoirr.remove(concepto)
    
    return supremoirr

In [58]:
supremo_irreducibles(df2)

[[['Avatar',
   'Avengers',
   'Inside Out 2',
   'LOTR',
   'Spiderman',
   'Star Wars VII',
   'Titanic'],
  ['Drama']],
 [['Avatar',
   'Avengers',
   'Jurassic World',
   'LOTR',
   'Spiderman',
   'Star Wars VII'],
  ['Acción']],
 [['Avatar',
   'Jurassic World',
   'LOTR',
   'Ne Zha',
   'Star Wars VII',
   'Zootopia 2'],
  ['Aventura']],
 [['Avatar', 'Avengers', 'Jurassic World', 'Spiderman', 'Star Wars VII'],
  ['Acción', 'Ci-Fi']],
 [['Inside Out 2', 'LOTR', 'Ne Zha', 'Zootopia 2'], ['Fant']],
 [['Inside Out 2', 'Ne Zha', 'Zootopia 2'], ['Anim', 'Fant']],
 [['Avatar', 'Ne Zha', 'Titanic'], ['Rom']],
 [['Avengers', 'Spiderman'], ['Acción', 'Ci-Fi', 'Superh', 'Drama']]]

In [60]:
for objeto in df.index:
    print(f"{objeto}: {intension(df, [objeto])}")

Avatar: ['Acción', 'Aventura', 'Ci-Fi', 'Rom', 'Drama']
Avengers: ['Acción', 'Ci-Fi', 'Superh', 'Drama']
Ne Zha: ['Aventura', 'Anim', 'Rom', 'Fant']
Star Wars VII: ['Acción', 'Aventura', 'Ci-Fi', 'Drama']
Spiderman: ['Acción', 'Ci-Fi', 'Superh', 'Drama']
LOTR: ['Acción', 'Aventura', 'Fant', 'Drama']
